# NutriCasa 2050 — Analise consolidada (card S5-10)

Todos os graficos e as duas consultas principais num lugar so, rodando
do comeco ao fim sem erro. E o que o professor abre se quiser conferir
de onde veio cada numero da apresentacao.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt

from nutricasa2050.analise.graficos import (
    grafico_pareto_custo_aderencia,
    grafico_pareto_custo_desperdicio,
    grafico_regiao_viavel,
    varrer_peso,
)
from nutricasa2050.analise.grafos import (
    componentes_conexas,
    cruzar_orfaos_com_desperdicio,
    grafo_de_substituicao,
    montar_grafo_bipartido,
)
from nutricasa2050.dados.carga import carregar_catalogo
from nutricasa2050.dominio import Parametros
from nutricasa2050.persistencia.banco import ConfiguracaoBanco, conexao, criar_engine, criar_schema
from nutricasa2050.simulacao.monte_carlo import rodar_monte_carlo

catalogo = carregar_catalogo()
print(f"{catalogo.total_receitas()} receitas, {catalogo.total_ingredientes()} ingredientes, "
      f"{len(catalogo.membros)} membros")


24 receitas, 50 ingredientes, 4 membros


## 1. Fronteira de Pareto: custo x desperdicio

Rodando o modelo varias vezes com pesos diferentes para o desperdicio,
aparece uma curva: **reduzir desperdicio custa dinheiro**. Nao existe
almoco gratis — e escolha, nao magica.


In [2]:
parametros_pareto = Parametros(horizonte=2, refeicoes_por_dia=1, orcamento=200.0, dias_compra=(1,))
pontos_desperdicio = varrer_peso(catalogo, parametros_pareto, "desperdicio", [0.5, 1.0, 2.0, 5.0, 10.0])
for ponto in pontos_desperdicio:
    print(ponto)

grafico_pareto_custo_desperdicio(pontos_desperdicio, "../figures/pareto_custo_desperdicio.png")
plt.show()
# Nota honesta: com este catalogo/horizonte, o desperdicio ja sai no
# minimo possivel independente do peso — a curva fica plana. A relacao
# matematica (NUNCA aumentar o desperdicio ao subir o peso) e verificada
# de verdade, com um catalogo desenhado para ter tradeoff genuino, em
# tests/unidade/test_graficos.py::test_pareto_e_monotonico.


{'peso': 0.5, 'status': 'Optimal', 'custo': 162.5, 'desperdicio': 28.5, 'desvio': 12453.745746999999}
{'peso': 1.0, 'status': 'Optimal', 'custo': 162.5, 'desperdicio': 28.5, 'desvio': 12453.745746999999}
{'peso': 2.0, 'status': 'Optimal', 'custo': 162.5, 'desperdicio': 28.5, 'desvio': 12453.745746999999}
{'peso': 5.0, 'status': 'Optimal', 'custo': 162.5, 'desperdicio': 28.5, 'desvio': 12453.745746999999}
{'peso': 10.0, 'status': 'Optimal', 'custo': 162.5, 'desperdicio': 28.5, 'desvio': 12453.745746999999}


## 2. Fronteira de Pareto: custo x aderencia nutricional

A mesma troca, vista por outro angulo: seguir a dieta a risca (minimizar
o desvio do alvo) tambem custa dinheiro.


In [3]:
pontos_desvio = varrer_peso(catalogo, parametros_pareto, "desvio", [0.5, 1.0, 2.0, 5.0])
grafico_pareto_custo_aderencia(pontos_desvio, "../figures/pareto_custo_aderencia.png")
plt.show()


## 3. Distribuicao das tres estrategias nos cenarios da Monte Carlo

Le a rodada OFICIAL publicada em `var/results.db` (100 cenarios x 3
estrategias, horizonte de 5 dias, orcamento R$ 500, tempo limite 30 s —
gerada com `make resultados` e copiada com `make publicar-resultados`).
E daqui que saem os numeros de `docs/resultados.md` (Sprint 6): **77,1% de
reducao de desperdicio** do otimizado contra a familia normal.

Se o arquivo nao existir (clone sem ele), o notebook roda uma Monte Carlo
PEQUENA em memoria (10 cenarios, horizonte de 1 dia) so para mostrar o
mecanismo — nesse caso e normal o desperdicio sair perto de zero, porque
quase nada vence em 1 dia.


In [ ]:
caminho_resultados = Path.cwd().parent / "var" / "results.db"
cfg = ConfiguracaoBanco(
    ambiente="development",
    url_banco_principal=f"sqlite:///{caminho_resultados}",
    url_banco_teste="sqlite:///:memory:",
)
if caminho_resultados.exists():
    engine = criar_engine(cfg, cfg.url_banco_principal)
    print(f"usando a rodada oficial: {caminho_resultados}")
else:
    engine = criar_engine(cfg, "sqlite:///:memory:")
    criar_schema(engine)
    parametros_mc = Parametros(horizonte=1, refeicoes_por_dia=3, orcamento=200.0, dias_compra=(1,))
    resumo_mc = rodar_monte_carlo(engine, catalogo, parametros_mc, n_cenarios=10)
    print("var/results.db ausente — Monte Carlo pequena de demonstracao:", resumo_mc)


In [5]:
from nutricasa2050.analise.graficos import grafico_desperdicio_por_cenario

grafico_desperdicio_por_cenario(engine, "../figures/desperdicio_por_cenario.png")
plt.show()


## 4. As duas consultas principais, com o SQL visivel

### 4.1 Reducao de desperdicio por estrategia


In [6]:
with conexao(engine) as conn:
    linhas = conn.execute(
        "SELECT estrategia, AVG(total_descarte) AS media_desperdicio "
        "FROM execucao GROUP BY estrategia ORDER BY media_desperdicio"
    ).fetchall()

for linha in linhas:
    print(f"{linha['estrategia']:>10}: R$ {linha['media_desperdicio']:.2f} em media")


    guloso: R$ 0.00 em media
    humano: R$ 0.00 em media
 otimizado: R$ 0.00 em media


### 4.2 Qual regra da nutricionista a realidade mais atropela

(a consulta da coluna `folga`, S3-06/S4-06) — agrupa a tabela
`aderencia` por membro e nutriente, contando em quantos
cenarios/execucoes a folga foi maior que zero.


In [7]:
with conexao(engine) as conn:
    linhas = conn.execute(
        "SELECT membro, regra AS nutriente, COUNT(*) AS vezes_com_folga, "
        "AVG(folga) AS folga_media "
        "FROM aderencia WHERE folga > 0 "
        "GROUP BY membro, regra "
        "ORDER BY vezes_com_folga DESC, folga_media DESC"
    ).fetchall()

if linhas:
    for linha in linhas:
        print(
            f"{linha['membro']:>18} / {linha['nutriente']:<12}: "
            f"{linha['vezes_com_folga']} vezes, folga media {linha['folga_media']:.1f}"
        )
else:
    print("Nenhuma folga usada nesta rodada pequena (catalogo real, horizonte curto).")


  idoso_hipertenso / kcal        : 21 vezes, folga media 257.3
    crianca_8_anos / carboidrato_g: 21 vezes, folga media 42.6
 adulto_sedentario / kcal        : 20 vezes, folga media 336.1
    crianca_8_anos / proteina_g  : 20 vezes, folga media 17.6
 adulto_sedentario / carboidrato_g: 19 vezes, folga media 86.0
  idoso_hipertenso / carboidrato_g: 19 vezes, folga media 65.2
 adulto_sedentario / proteina_g  : 19 vezes, folga media 33.6
  idoso_hipertenso / gordura_g   : 19 vezes, folga media 5.4
      adulto_ativo / kcal        : 12 vezes, folga media 711.4
    crianca_8_anos / kcal        : 12 vezes, folga media 406.7
      adulto_ativo / carboidrato_g: 12 vezes, folga media 113.3
      adulto_ativo / gordura_g   : 12 vezes, folga media 18.5
    crianca_8_anos / gordura_g   : 12 vezes, folga media 10.1
  idoso_hipertenso / fibra_g     : 12 vezes, folga media 5.7
    crianca_8_anos / sodio_mg    : 11 vezes, folga media 802.0
      adulto_ativo / proteina_g  : 11 vezes, folga media 13.7


## 5. O desenho da geometria (regiao viavel em 2D)

Uma versao minuscula do modelo (so duas variaveis), desenhada: a area
sombreada e a regiao viavel, as linhas pontilhadas sao curvas de nivel
do custo, e o ponto marcado e o otimo — que SEMPRE cai num vertice
(teorema fundamental da programacao linear, card S5-03).


In [8]:
grafico_regiao_viavel("../figures/regiao_viavel.png")
plt.show()


## 6. Mapa de ingredientes (grafo bipartido) e ingredientes orfaos

Ligando receitas e ingredientes aparece um grafo bipartido. Os
ingredientes orfaos (usados por menos de 2 receitas) sao justamente os
que mais viram lixo — a secao final cruza essa previsao teorica com o
desperdicio REAL observado na Monte Carlo acima.


In [9]:
grafo = montar_grafo_bipartido(catalogo)
graus_receitas = {no: len(viz) for no, viz in grafo.items() if no.startswith("r:")}
graus_ingredientes = {no: len(viz) for no, viz in grafo.items() if no.startswith("i:")}
print(f"{len(graus_receitas)} receitas, {len(graus_ingredientes)} ingredientes, "
      f"{sum(graus_receitas.values())} ligacoes")

orfaos = catalogo.ingredientes_orfaos()
print(f"{len(orfaos)} ingredientes orfaos: {orfaos}")


24 receitas, 50 ingredientes, 178 ligacoes
0 ingredientes orfaos: []


In [10]:
substituicao = grafo_de_substituicao(catalogo)
componentes = componentes_conexas(substituicao)
print(f"{len(componentes)} grupos de receitas que podem se substituir sem gerar sobra")
print("maior grupo:", max(componentes, key=len))


1 grupos de receitas que podem se substituir sem gerar sobra
maior grupo: {'sanduiche_quente_queijo_tomate', 'arroz_feijao_ovo', 'frango_grelhado_batata', 'bolo_banana_aveia', 'arroz_de_forno_legumes', 'arroz_carreteiro', 'salada_frango_maca', 'macarrao_frango_tomate', 'torta_legumes', 'iogurte_aveia_maca', 'farofa_de_ovo', 'sopa_legumes_frango', 'panqueca_banana_aveia', 'macarrao_atum_ervilha', 'escondidinho_batata_carne', 'feijao_com_calabresa', 'creme_cenoura', 'omelete_legumes', 'frango_ao_molho_tomate', 'peixe_assado_pure', 'tilapia_assada_salada', 'salada_repolho_maca', 'arroz_ovo', 'feijao_arroz_couve'}


### Teoria confirmada pelo dado real

Os ingredientes orfaos, ordenados pelo desperdicio REAL observado na
Monte Carlo rodada acima — e a funcao que transforma teoria de grafos
em resultado.

**Nota honesta:** o catalogo atual de dados (`data/`) nao tem nenhum
ingrediente orfao — eles foram encontrados e corrigidos no card S1-13
(ver `docs/resultados.md`, Sprint 1). Esta lista, entao, sai vazia, o
que e o resultado CORRETO e esperado, nao um bug. A funcao esta testada
de verdade com um catalogo que TEM orfaos em
`tests/unidade/test_grafos.py::test_cruzar_orfaos_com_desperdicio_ordena_pelo_total_real`.


In [11]:
cruzamento = cruzar_orfaos_com_desperdicio(engine, catalogo)
for item in cruzamento:
    print(f"{item['nome']:<20} desperdicio observado: {item['desperdicio_observado']:.2f}")

if engine.conexao_compartilhada is not None:
    engine.conexao_compartilhada.close()
